# Tuần 08: Learner Error Analysis

Mục tiêu: biến câu trả lời của người học thành bằng chứng có thể đọc được trong paper. Tuần này học cách đếm lỗi, loại off-prompt rows, làm bảng chéo, chọn ví dụ đại diện và viết Results paragraph.


## Trước khi chạy code

Cell setup bên dưới chuẩn bị thư viện, kiểm tra file data và tạo folder output. Bạn chỉ cần **Run** cell này trước; chưa cần hiểu từng dòng.

- Cell này làm gì? Import thư viện, tìm folder Week 08, kiểm tra SHA-256 của CSV.
- Kết quả mong đợi: in ra `Week folder`, `Data file`, và `SHA-256`.
- Nếu lỗi: kiểm tra đã mở notebook từ project root hoặc folder Week 08; nếu thiếu package, chạy `python -m pip install -r requirements.txt`.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

try:
    import pandas as pd
    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError as error:
    raise ImportError("Install course packages first: python -m pip install -r requirements.txt") from error

EXPECTED_SHA256 = "0f17314ecc223b8a0228f46212949ba0bc4b56aeca6848b891e9c00cc263d3d4"
RAW_URL = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-08-learner-error-analysis/data/raw/week08_learner_error_coding.csv"


def find_week_dir(start=Path.cwd()):
    candidates = [start, *start.parents]
    for candidate in candidates:
        direct = candidate / "data/raw/week08_learner_error_coding.csv"
        nested = candidate / "weeks/week-08-learner-error-analysis/data/raw/week08_learner_error_coding.csv"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "weeks/week-08-learner-error-analysis"
    fallback = Path("weeks/week-08-learner-error-analysis")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback

WEEK_DIR = find_week_dir()
DATA_PATH = WEEK_DIR / "data/raw/week08_learner_error_coding.csv"
TABLE_DIR = WEEK_DIR / "outputs/tables"
FIGURE_DIR = WEEK_DIR / "outputs/figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(RAW_URL, DATA_PATH)

actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA256:
    raise ValueError(f"Raw data changed. Expected {EXPECTED_SHA256}, got {actual_sha}")

print(f"Week folder: {WEEK_DIR}")
print(f"Data file: {DATA_PATH.relative_to(WEEK_DIR)}")
print(f"SHA-256: {actual_sha}")


Week folder: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-08-learner-error-analysis
Data file: data/raw/week08_learner_error_coding.csv
SHA-256: 0f17314ecc223b8a0228f46212949ba0bc4b56aeca6848b891e9c00cc263d3d4


## 1. Research loop của tuần này

Ta dùng một vòng rất ngắn:

```text
learner response -> on-prompt filter -> error code -> frequency table -> representative examples -> Results paragraph
```

Điểm mới sau review: không phân tích dòng trả lời nhầm prompt trong crosstab chính. Một dòng phải vừa `usable_error == True` vừa `on_prompt == True`.


In [2]:
df = pd.read_csv(DATA_PATH)

analysis_rows = df[(df["usable_error"] == True) & (df["on_prompt"] == True)].copy()
error_rows = analysis_rows[analysis_rows["has_error"] == True].copy()
no_error_rows = analysis_rows[analysis_rows["has_error"] == False].copy()
excluded_rows = df[(df["usable_error"] != True) | (df["on_prompt"] != True)].copy()

for table in [analysis_rows, error_rows, no_error_rows]:
    table["severity"] = pd.to_numeric(table["severity"], errors="coerce")

print("Raw response rows:", len(df))
print("On-prompt usable rows:", len(analysis_rows))
print("Coded error rows:", len(error_rows))
print("No-error rows:", len(no_error_rows))
print("Excluded rows:", len(excluded_rows))
print("Excluded categories:")
print(excluded_rows["error_category"].value_counts().to_string())


Raw response rows: 80
On-prompt usable rows: 68
Coded error rows: 57
No-error rows: 11
Excluded rows: 12
Excluded categories:
error_category
off_prompt_response    8
unusable_response      4


## 2. Codebook tối thiểu

Không phải mọi dòng đều là lỗi. Ta giữ ba loại nhãn ngoài phân tích lỗi:

- `no_error`: câu trả lời target-like, không count là lỗi;
- `unusable_response`: blank/off-task, không phân tích;
- `off_prompt_response`: câu trả lời có thể đọc được nhưng không trả lời đúng prompt, không đưa vào crosstab target structure.


In [3]:
active_codebook = {
    "aspect_marker": "了 placement or aspect marking",
    "measure_word": "classifier choice or classifier omission",
    "result_complement": "missing/wrong result complement",
    "tone_marking": "pinyin tone mark problem",
    "word_order": "target word-order frame problem",
}

pd.DataFrame(
    [{"error_category": key, "teaching_meaning": value} for key, value in active_codebook.items()]
)


## 3. Frequency table: lỗi nào nhiều nhất?

`value_counts()` trả lời câu hỏi: nhãn lỗi nào xuất hiện nhiều nhất trong các coded error rows?


In [4]:
error_frequency = (
    error_rows["error_category"]
    .value_counts()
    .rename_axis("error_category")
    .reset_index(name="n")
)
error_frequency["percent_coded_errors"] = (error_frequency["n"] / len(error_rows) * 100).round(1)
error_frequency.to_csv(TABLE_DIR / "week08_error_frequency.csv", index=False)
error_frequency


## 4. Crosstab: lỗi nào gắn với target nào?

`pd.crosstab()` cho biết mỗi target structure có bao nhiêu dòng thuộc từng error category. Vì đã loại `off_prompt_response`, bảng này chỉ đọc các lỗi trong đúng prompt.


In [5]:
error_by_target = pd.crosstab(
    error_rows["target_structure"],
    error_rows["error_category"]
)
error_by_target.to_csv(TABLE_DIR / "week08_error_by_target_structure.csv")
error_by_target


## 5. Teaching priority không chỉ là nhiều hay ít

Ta dùng heuristic đơn giản: `priority_score = n * mean_severity`. Đây là gợi ý dạy lại, không phải bằng chứng nguyên nhân.


In [6]:
teaching_moves = {
    "aspect_marker": "Use short action-result routines and compare verb-final 了 with sentence-final 了.",
    "measure_word": "Teach object-specific classifier chunks such as 一杯茶, 两本书, 三张票.",
    "result_complement": "Contrast 找票 and 找到票了; ask whether the action reached a result.",
    "tone_marking": "Use minimal tone pairs and require tone marks in written rehearsal.",
    "word_order": "Practice subject + time/place + verb phrase frames before open production.",
}

priority_table = (
    error_rows.groupby("error_category")
    .agg(
        n=("response_id", "count"),
        mean_severity=("severity", "mean"),
        target_count=("target_structure", "nunique"),
    )
    .reset_index()
)
priority_table["priority_score"] = (priority_table["n"] * priority_table["mean_severity"]).round(2)
priority_table["mean_severity"] = priority_table["mean_severity"].round(2)
priority_table["teaching_move"] = priority_table["error_category"].map(teaching_moves)
priority_table = priority_table.sort_values(["priority_score", "n"], ascending=False)
priority_table.to_csv(TABLE_DIR / "week08_teaching_priority_table.csv", index=False)
priority_table


## 6. Representative examples

Ví dụ đại diện nên rõ prompt, learner answer, expected answer và teaching note. Không chọn `off_prompt_response` làm ví dụ chính.


In [7]:
representative_examples = (
    error_rows.sort_values(["error_category", "severity", "response_id"], ascending=[True, False, True])
    .groupby("error_category", as_index=False)
    .head(1)[[
        "error_category", "response_id", "target_structure", "prompt_vi",
        "learner_answer", "expected_answer", "error_feature", "correction_note"
    ]]
    .sort_values("error_category")
)
representative_examples.to_csv(TABLE_DIR / "week08_representative_examples.csv", index=False)
representative_examples


## 7. Figures cho paper

Figure 1 cho frequency. Figure 2 cho target structure by error category. Caption phải nêu N và limitation.


In [8]:
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(8.8, 5.2))
sns.barplot(data=error_frequency, y="error_category", x="n", color="#2563eb", ax=ax)
ax.set_title("Week 08 coded learner-error categories")
ax.set_xlabel("Coded error rows (n)")
ax.set_ylabel("Error category")
for i, row in error_frequency.reset_index().iterrows():
    ax.text(row["n"] + 0.15, i, f"{row['n']} ({row['percent_coded_errors']}%)", va="center", fontsize=10)
ax.set_xlim(0, max(error_frequency["n"]) + 4)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "week08_error_category_frequency.png", dpi=180)
fig.savefig(FIGURE_DIR / "week08_error_category_frequency.svg")
plt.close(fig)

fig, ax = plt.subplots(figsize=(9.4, 5.6))
sns.heatmap(error_by_target, annot=True, fmt="d", cmap="Blues", linewidths=.5, cbar_kws={"label":"coded error rows"}, ax=ax)
ax.set_title("Week 08 target structure by error category")
ax.set_xlabel("Error category")
ax.set_ylabel("Target structure")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "week08_error_by_target_heatmap.png", dpi=180)
fig.savefig(FIGURE_DIR / "week08_error_by_target_heatmap.svg")
plt.close(fig)

print("Saved figures:")
print(FIGURE_DIR / "week08_error_category_frequency.png")
print(FIGURE_DIR / "week08_error_by_target_heatmap.png")


Saved figures:
/Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-08-learner-error-analysis/outputs/figures/week08_error_category_frequency.png
/Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-08-learner-error-analysis/outputs/figures/week08_error_by_target_heatmap.png


## 8. Output kiểm tra nhanh

Các file dưới đây là những gì người học cần mở trước khi viết Results.


In [9]:
for path in sorted(TABLE_DIR.glob("week08_*.csv")):
    print(path.relative_to(WEEK_DIR))
for path in sorted(FIGURE_DIR.glob("week08_*.png")):
    print(path.relative_to(WEEK_DIR))


outputs/tables/week08_error_by_target_structure.csv
outputs/tables/week08_error_frequency.csv
outputs/tables/week08_representative_examples.csv
outputs/tables/week08_teaching_priority_table.csv
outputs/figures/week08_error_by_target_heatmap.png
outputs/figures/week08_error_category_frequency.png


## 9. Results paragraph frame

Đoạn dưới đây là draft mẫu. Hãy dùng số liệu của bạn, thêm một caption, và không viết broad claim về tất cả người học Việt Nam.


In [10]:
top_error = error_frequency.iloc[0]
second_error = error_frequency.iloc[1]
top_priority = priority_table.iloc[0]

paragraph = (
    f"In the on-prompt usable learner-response records (N = {len(analysis_rows)}), "
    f"{len(error_rows)} rows contained a coded learner error. "
    f"The most frequent category was {top_error['error_category']} "
    f"(n = {top_error['n']}, {top_error['percent_coded_errors']}% of coded errors), "
    f"followed by {second_error['error_category']} (n = {second_error['n']}). "
    f"The crosstab shows that each top category is tied to its practiced target after off-prompt responses are excluded. "
    f"The teaching-priority table ranks {top_priority['error_category']} highest because it combines frequency and severity. "
    f"Because the dataset is synthetic and small, the pattern should guide follow-up teaching design rather than support broad claims about Vietnamese learners of Chinese."
)
print(paragraph)
print("Word count:", len(paragraph.split()))


In the on-prompt usable learner-response records (N = 68), 57 rows contained a coded learner error. The most frequent category was tone_marking (n = 13, 22.8% of coded errors), followed by result_complement (n = 11). The crosstab shows that each top category is tied to its practiced target after off-prompt responses are excluded. The teaching-priority table ranks tone_marking highest because it combines frequency and severity. Because the dataset is synthetic and small, the pattern should guide follow-up teaching design rather than support broad claims about Vietnamese learners of Chinese.
Word count: 89


## 10. Exercise

1. Run the notebook from top to bottom.
2. Open `outputs/tables/week08_error_frequency.csv` and identify the top error category.
3. Open `outputs/tables/week08_error_by_target_structure.csv` and explain one high cell.
4. Open `outputs/tables/week08_representative_examples.csv` and choose two examples for your Results paragraph.
5. Write two figure captions, one for each figure.
6. Write a 120-160 word Results paragraph.
7. Add one source note: which reading helps you connect errors to teaching targets?
8. Stretch: create `error_rows_stretch = error_rows.copy()`, recode `R004` from `word_order` to `word_order_target_frame`, and explain why the new code is clearer. Do not edit the raw CSV because the notebook checks SHA-256.
